Copyright (c) European Space Agency, 2025.  
This file is subject to the terms and conditions defined in file 'LICENCE.txt',   
which is part of this source code package. No part of the package, including  
this file, may be copied, modified, propagated, or distributed except according to   
the terms contained in the file ‘LICENCE.txt’.

# Analysis of simulated Gaia measurement timeseries in the context of the ESA workshop "Exploring Tatooine and Beyond: Circumbinary Planets with ESA Missions" held at ESAC in December 2025

## Workshop website: https://www.cosmos.esa.int/web/circumbinary-planets-with-esa-missions/home

## Jupyter notebook basics: https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Running%20Code.html

## This notebook was prepared by the Gaia collaboration. If you find it useful for your research or otherwise, please acknowledge the Gaia mission as indicated at https://www.cosmos.esa.int/web/gaia-users/credits 

## Content of this notebook  
This notebook demonstrates the following steps:
- Load the simulated Gaia DR4-like timeseries data of the Gaia astrometric instrument for a given source. 
- Fit a single Keplerian orbit to the astrometric data
- Inspect residuals
- Fit additional Keplerian signals if needed

## Environment installation
- This notebook is most easily executed within ESA Datalabs

### Imports

In [ ]:
from collections import OrderedDict
import copy
import os

from astropy.time import Time
import astropy.units as u
from IPython.display import display, Markdown
import matplotlib.pyplot as pl
import numpy as np
import pandas as pd

from kepmodel.astro import AstroModel as AstrometricModel
from kepmodel.rv import RvModel
from kepmodel.astrorv import AstroRvModel
from pystrometry.pystrometry import convert_from_angular_to_linear, pjGet_m2, OrbitSystem, MS_kg, MJ_kg, convert_from_linear_to_angular
from pystrometry import gaia_astrometry as gaia_astrometry_plotter
import spleaf

### Folder and constants definition

In [ ]:
base_dir = os.getcwd()
print(f"Base directory: {base_dir}")
data_dir = os.path.join(base_dir)
plot_dir = os.path.join(base_dir)

DR4_REFERENCE_EPOCH = Time('2017.5', format='jyear', scale='tcb')

### Selected source name and parameters (please edit this cell as needed)

In [ ]:
target_name='Gaia-4' # host of Gaia-4 b
# target_name='Kepler-16'
# target_name = 'HD202206'
# target_name = 'unknown'

m1_MS = 1.0 # default primary mass in Msun



## Load astrometric-timeseries data

In [ ]:
data_file = os.path.join(base_dir, f'gaia_simulated_astrometry_{target_name}.csv')
gaia_astrometry = pd.read_csv(data_file, comment='#')

ra_deg = gaia_astrometry['ra0'][0] # Right ascension of reference point
dec_deg = gaia_astrometry['dec0'][0] # Declination of reference point
source_id = gaia_astrometry['source_id'][0]

# rename soem fields and set auxiliary fields
gaia_astrometry['transitid'] = gaia_astrometry['transit_id']
gaia_astrometry['relative_time_year'] = Time(gaia_astrometry['obs_time_tcb'], format='jd', scale='tcb').jyear - DR4_REFERENCE_EPOCH.jyear
gaia_astrometry['mjd'] = Time(gaia_astrometry['obs_time_tcb'], format='jd', scale='tcb').mjd
gaia_astrometry['relative_time_day'] = gaia_astrometry['relative_time_year'] * u.year.to(u.day)
gaia_astrometry['cpsi_obs'] = np.cos(np.deg2rad(gaia_astrometry['scan_pos_angle']))
gaia_astrometry['spsi_obs'] = np.sin(np.deg2rad(gaia_astrometry['scan_pos_angle']))
gaia_astrometry['ppfact_obs'] = gaia_astrometry['parallax_factor_al']
gaia_astrometry['da_mas'] = gaia_astrometry['centroid_pos_al']
gaia_astrometry['sigma_da_mas'] = gaia_astrometry['centroid_pos_error_al']

print(gaia_astrometry.columns)

### Display a few data records

In [ ]:
columns_to_show = 'source_id transit_id ccd_id relative_time_year centroid_pos_al centroid_pos_error_al parallax_factor_al scan_pos_angle'.split()
display(gaia_astrometry.query('ccd_id == 1')[columns_to_show].sample(10, random_state=1).sort_values('relative_time_year'))

In [ ]:
# display histogram of one field
col = 'centroid_pos_error_al'
gaia_astrometry[col].hist(bins=20)
pl.xlabel('centroid_pos_error_al (mas)')


## Fit the standard astrometric model 
The standard astrometric model for the motion $w_\mathrm{ss}$ of a single star can be written as
\begin{equation}
w_\mathrm{ss} = ( \Delta\alpha^{\star} + \mu_{\alpha^\star} \, t ) \, \sin \psi + ( \Delta\delta + \mu_\delta \, t ) \, \cos \psi + \varpi \, f_\varpi, 
\end{equation}
where $\Delta\alpha^{\star} = \Delta\alpha \cos{\delta}$ and $\Delta \delta$ are small offsets in equatorial coordinates from a fixed reference point ($\alpha_0$, $\delta_0$), $\mu_{\alpha^\star}$ and $\mu_\delta$ are proper motions in these coordinates, $t$ is time since the Gaia DR4 reference time J2017.5, $\varpi$ is the parallax, $f_\varpi$ is the parallax factor, and $\psi$ is the instantaneous Gaia scan angle. For more details see Section 3 of https://www.aanda.org/articles/aa/full_html/2023/06/aa44161-22/aa44161-22.html and https://www.aanda.org/articles/aa/full_html/2023/06/aa43969-22/aa43969-22.html

We use the open-source python package [kepmodel](https://obswww.unige.ch/~delisle/kepmodel/doc/) (Delisle & Ségransan, 2022, A&A, 667, A172), which allows us to model radial velocity and/or astrometric time series including Keplerian orbits.

In [ ]:
include_jitter_term = False

if include_jitter_term:
    astrometric_jitter_value = 0.05
else:
    astrometric_jitter_value = 0.0
    
# set up the single-star model with an additional jitter term of 0.05 mas
single_star_model = AstrometricModel(gaia_astrometry['relative_time_day'].values, 
                                     gaia_astrometry['centroid_pos_al'].values, 
                                     gaia_astrometry['cpsi_obs'].values, 
                                     gaia_astrometry['spsi_obs'].values, 
                                     err=spleaf.term.Error(gaia_astrometry['centroid_pos_error_al'].values),
                                     jit=spleaf.term.Jitter(astrometric_jitter_value))
    
# define the linear parameters
single_star_model.add_lin(gaia_astrometry['spsi_obs'].values, 'ra')
single_star_model.add_lin(gaia_astrometry['cpsi_obs'].values, 'dec')
single_star_model.add_lin(gaia_astrometry['parallax_factor_al'].values, 'parallax')
single_star_model.add_lin(gaia_astrometry['relative_time_year'].values * gaia_astrometry['spsi_obs'].values, 'mura')
single_star_model.add_lin(gaia_astrometry['relative_time_year'].values * gaia_astrometry['cpsi_obs'].values, 'mudec')

# add jitter term
if include_jitter_term:
    single_star_model.fit_param += ['cov.jit.sig']

# perform the fit
single_star_model.fit()

print("Best-fit single-star model parameters: (the units are mas and mas/year)")
single_star_model.show_param()

## Fit the Keplerian astrometric model 

Using [kepmodel](https://obswww.unige.ch/~delisle/kepmodel/doc/), we first compute the periodogram of the residuals of the standard-model fit. With the estimated period we initialise a Keplerian model and fit it to the data.

The astrometric motion corresponding to a Keplerian orbit of a binary system has generally seven independent parameters. These
are the period $P$, the epoch of periastron passage $T_0$, the eccentricity $e$, the inclination $i$, the ascending node $\Omega$, the argument of periastron $\omega$, and the semi-major axis of the photocentre $a_0$. The Thiele-Innes coefficients $A, B, F, G$, which linearise part of the equations, are defined as
\begin{eqnarray}
%\begin{split}
% \begin{array}{ll}
A &=&  \ \ \, a_0 \; (\cos \omega \cos \Omega - \sin \omega \sin \Omega \cos i)   \\
B &=&  \ \ \, a_0 \; (\cos \omega \sin \Omega + \sin \omega \cos \Omega \cos i)  \\
F &=& -a_0 \; (\sin \omega \cos \Omega + \cos \omega \sin \Omega \cos i)  \\
G &=&  -a_0 \; (\sin \omega \sin \Omega - \cos \omega \cos \Omega \cos i) 
% \end{array}
%\end{split}
\end{eqnarray}

The elliptical rectangular coordinates $X$ and $Y$ are functions of eccentric anomaly $E$ and eccentricity: 
\begin{eqnarray}
E - e \sin E &=& \frac{2\pi}{P} (t-T_0)\\
X &=& \cos E - e\\
Y &=& \sqrt{1-e^2} \sin E
\end{eqnarray}

The single Keplerian model $w_\mathrm{k1}$ can then be written as 
\begin{equation}
w_\mathrm{k1} = (B \, X + G \, Y) \sin \psi + (A \, X + F \, Y) \cos \psi.
\end{equation}

The combined model $w^\mathrm{(model)}$ for the Gaia along-scan abscissa is
\begin{equation}
\begin{split}
w^\mathrm{(model)} =&\, w_\mathrm{ss} + w_\mathrm{k1} \\
 =&\, ( \Delta\alpha^{\star} + \mu_{\alpha^\star} \, t ) \, \sin \psi + ( \Delta\delta + \mu_\delta \, t ) \, \cos \psi + \varpi \, f_\varpi \\
 &+\, (B \, X + G \, Y) \sin \psi + (A \, X + F \, Y) \cos \psi.
\end{split}\end{equation}


More details are given in Section 3 of https://www.aanda.org/articles/aa/full_html/2023/06/aa44161-22/aa44161-22.html and in https://www.aanda.org/articles/aa/full_html/2023/06/aa43969-22/aa43969-22.html

In [ ]:
model = copy.deepcopy(single_star_model)

# Periodogram settings 
Pmin = 5
Pmax = 10000
nfreq = 10000
nu0 = 2 * np.pi / Pmax
dnu = (2 * np.pi / Pmin - nu0) / (nfreq - 1)

# plot the single-star residuals
resisuals = model.residuals()
res_err = np.sqrt(model.cov.A)
pl.figure(figsize=(10, 3))
pl.errorbar(model.t, resisuals, yerr=res_err, fmt='o', ecolor='0.7', ms=4, color='k')
pl.xlabel(f'Relative time (days)')
pl.ylabel('Single-star residuals (mas)')
pl.axhline(ls='--')
pl.show()    

# compute periodogram
nu, power = model.periodogram(nu0, dnu, nfreq)

# convert from angular frequency to period
P = 2 * np.pi / nu

# identify highest peak and compute false-alarm probability (FAP)
kmax = np.argmax(power)
faplvl = model.fap(power[kmax], nu.max())

# plot periodogram as a function of frequency
pl.figure(figsize=(10, 3))
pl.plot(nu, power, 'k', lw=1.2)
pl.ylim(0, 1.1 * power.max())
pl.xlabel('Angular frequency (rad/day)')
pl.ylabel('Normalized power')
pl.title(f'nu={nu[kmax]:.4f} rad/d' + 20*' ' + f'FAP={faplvl:.2g}')
pl.show()

# plot periodogram as a function of period
pl.figure(figsize=(10, 3))
pl.plot(2 * [P[kmax]], [0, 1.1 * power.max()], 'r')
pl.plot(P, power, 'k', lw=1.2, rasterized=True)
pl.xlim(Pmin, Pmax)
pl.ylim(0, 1.1 * power.max())
pl.xscale('log')
pl.xlabel('Period (days)')
pl.ylabel('Normalized power')
pl.title(f'P={P[kmax]:.1f} d' + 20*' ' + f'FAP={faplvl:.2g}')
pl.show()

## Fit the Keplerian model

In [ ]:
keplerian_model = copy.deepcopy(model)

print(f'Adding a Keplerian with period {P[kmax]:.1f} days to model.\n')

keplerian_model.add_keplerian_from_period(P[kmax])

# fit the new model
keplerian_model.fit()

print(f"Best-fit parameter (Thiele-Innes coefficients)")
keplerian_model.show_param()

# print('Change orbital parameter set:')
param = ['P', 'Tp', 'as', 'e', 'w', 'i', 'bigw']
keplerian_model.set_keplerian_param(f'0', param=param)

print(f"Best-fit parameter (Campbell elements)\nkep.0.P is the period in days, kep.0.as is the semimajor axis in milli-arcseconds\n")
keplerian_model.show_param()    

## Compute the mass of the companion

In [ ]:
import astropy.units as u

keplerian_parameters = {}
for i, key in enumerate(keplerian_model.keplerian['0']._param):
    keplerian_parameters[key] = keplerian_model.keplerian['0']._par[i]

linear_parameters = {}
for i, key in enumerate(keplerian_model._lin_name):
    linear_parameters[key] = keplerian_model._lin_par[i]

a_m = convert_from_angular_to_linear(keplerian_parameters['as'], linear_parameters['parallax'])
m2_kg = pjGet_m2(m1_MS * MS_kg, a_m, keplerian_parameters['P'])
m2_MS = m2_kg / MS_kg
mass_function = keplerian_parameters['as']**3 * (u.year.to(u.day))**2 / (keplerian_parameters['P']**2 * linear_parameters['parallax']**3)

display(Markdown(f'### <strong>Results from astrometry only:</strong>'))
display(Markdown(f'### <strong>The mass function value is {mass_function:.1e} solar masses</strong>'))
display(Markdown(f'### <strong>With a primary mass of {m1_MS} solar masses, the companion has a mass of {m2_MS:.3f} solar masses ({m2_MS*u.Msun.to(u.Mjupiter):.1f} Jupiter masses)</strong><br/>'))

## Create orbit model and display it together with the Gaia measurements

We use plotting tools provided by the open-source python package [pystrometry](https://github.com/Johannes-Sahlmann/pystrometry) (https://zenodo.org/records/10839734), which supports the analysis of high-precision astrometry timeseries. 

In [ ]:
# A helper function for plotting
def plot_astrometry(astrometry_timeseries, parameter_df, plot_dir, reference_time, model_name):
    """Make a figure showing the model and the data."""
    
    source_id = parameter_df.loc[0, 'source_id']

    iad = gaia_astrometry_plotter.GaiaValIad.from_dataframe(astrometry_timeseries, source_id, 'source_id')

    iad._mjd_field = 'mjd'
    iad.time_column = 'relative_time_year'
    iad.set_reference_time(reference_time)

    parameter_dict = parameter_df[parameter_df['source_id'] == source_id].iloc[0].to_dict()
    parameter_dict['source_id'] = np.int64(source_id)
    parameter_dict['name_seed'] = f"{source_id}_{model_name}"
    if model_name == 'multiple_star':
        axp = gaia_astrometry_plotter.plot_individual_orbit(parameter_dict, iad, plot_dir=plot_dir, m1_MS=parameter_dict['m1_MS'])
    return axp
    
reference_time = DR4_REFERENCE_EPOCH
model_name = 'multiple_star'

attribute_dict = OrderedDict([  
                                ('ra', ra_deg), 
                                ('dec', dec_deg),    
                                ('absolute_plx_mas', linear_parameters['parallax']), 
                                ('muRA_mas', linear_parameters['mura']),
                                ('muDE_mas', linear_parameters['mudec']),
                                ('muAlphaStar_masPyr', linear_parameters['mura']),
                                ('muDelta_masPyr', linear_parameters['mudec']),
                                ('Tref_MJD', reference_time.mjd),
                                ('scan_angle_definition', 'gaia'),
                             ])  

attribute_dict['m2_MJ'] = m2_kg / MJ_kg
attribute_dict['m1_MS'] = m1_MS
attribute_dict['P_day'] = keplerian_parameters['P']
attribute_dict['p1_omega_deg'] = np.rad2deg(keplerian_parameters['w'])
attribute_dict['ecc'] = keplerian_parameters['e']
attribute_dict['p1_OMEGA_deg'] = np.rad2deg(keplerian_parameters['bigw'])
attribute_dict['p1_incl_deg'] = np.rad2deg(keplerian_parameters['i'])
attribute_dict['p1_Tp_day'] = attribute_dict['Tref_MJD'] + keplerian_parameters['Tp']  # this is in MJD
attribute_dict['p1_a1_mas'] = keplerian_parameters['as']
attribute_dict['p1_ecc'] = keplerian_parameters['e']
attribute_dict['p1_period_day'] = keplerian_parameters['P']
attribute_dict['p1_Tp_day-T0'] = keplerian_parameters['Tp']
attribute_dict['offset_alphastar_mas'] = linear_parameters['ra']
attribute_dict['offset_delta_mas'] = linear_parameters['dec']

attribute_dict['source_id'] = source_id

attribute_dict['epoch_omc_description'] = ''
attribute_dict['orbit_signal_description'] = ''
attribute_dict['orbit_description'] = [['']]
attribute_dict['ppm_description'] = ''
attribute_dict['title'] = f'{target_name} astrometric orbit'
attribute_dict['frame_omc_description'] = ''
attribute_dict['orbit_show_frame_data'] = False

display(Markdown(f'## <strong>The five figure panels below show the following (for a detailed description see https://www.aanda.org/articles/aa/full_html/2023/06/aa44161-22/F12.html):\nTop left: Parallax and proper motion of the source projected on the sky  \nBottom left: On-sky orbital signature relative to the system\'s barycentre   \nTop right: Orbital signature relative to the system barycentre as a function of time  \nMiddle right: Aggregated residuals as a function of time  \nBottom right: Individual residuals as a function of time </strong>'))
axp = plot_astrometry(gaia_astrometry, pd.DataFrame(attribute_dict, index=[0]), plot_dir, reference_time, model_name)    

## Look for periodicity in the residuals of the single-Keplerian model fit

In [ ]:
multiple_companion_model = copy.deepcopy(keplerian_model)

fap_threshold = 1e-3

# Periodogram settings
# Pmin = 5
# Pmax = 10000
# nfreq = 10000
# nu0 = 2 * np.pi / Pmax
# dnu = (2 * np.pi / Pmin - nu0) / (nfreq - 1)


for kpla in range(3):
    res = multiple_companion_model.residuals()
    res_err = np.sqrt(multiple_companion_model.cov.A)
    pl.figure()
    pl.errorbar(multiple_companion_model.t,
      res,
      yerr=res_err,
      fmt='.',
      rasterized=True)
    # pl.xlabel(f'BJD - {epoch_bjd}')
    pl.ylabel('res. (mas)')
    pl.show()    
    nu, power = multiple_companion_model.periodogram(nu0, dnu, nfreq)
    P = 2 * np.pi / nu
    
    kmax = np.argmax(power)
    faplvl = multiple_companion_model.fap(power[kmax], nu.max())

    pl.figure(figsize=(10, 3))
    pl.plot(2 * [P[kmax]], [0, 1.1 * power.max()], 'r')
    pl.plot(P, power, 'k', lw=1.2, rasterized=True)
    pl.xlim(Pmin, Pmax)
    pl.ylim(0, 1.1 * power.max())
    pl.xscale('log')
    pl.xlabel('Period (d)')
    pl.ylabel('Normalized power')
    pl.title(f'P={P[kmax]:.3f} d' + 20*' ' + f'FAP={faplvl:.2g}')
    # pl.axhline(fap_threshold, ls='--', label='fap_threshold')
    pl.show()
    
    if faplvl > fap_threshold:
        print('No more significant peak')
        break
    print(f'Add a planet at {P[kmax]:.3f} d:')
    multiple_companion_model.add_keplerian_from_period(P[kmax])
    multiple_companion_model.fit()
    multiple_companion_model.show_param()
    
    print('\nChange orbital parameter set:')
    param = ['P', 'Tp', 'as', 'e', 'w', 'i', 'bigw']
    multiple_companion_model.set_keplerian_param(f'{kpla}', param=param)
    multiple_companion_model.show_param()    
    # print('loglikelihood = {}\n'.format(model.loglike()))